# Appendix Cancer Prediction — EDA & Visualization
**Author:** Jyotsana Sharma  
**Role:** EDA + Visualization + Insights  
**Dataset:** 260,000 patient records, 25 features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Consistent styling
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'axes.titleweight': 'bold'})

CANCER_COLORS = {'Yes': '#e05c5c', 'No': '#5c9ae0'}
DATA_PATH = 'appendix_cancer_prediction_dataset.csv'

## 1. Load & Initial Inspection

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')
print(f'\n=== Target Distribution ===')
print(df['Appendix_Cancer_Prediction'].value_counts())
print(df['Appendix_Cancer_Prediction'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

In [ ]:
df.describe(include='all').T

## 2. Feature Engineering (for analysis)

In [ ]:
# Age groups
df['Age_Group'] = pd.cut(df['Age'], bins=[0,20,35,50,65,80,120],
                          labels=['<20','20-35','36-50','51-65','66-80','80+'])

# BMI categories (WHO standard)
df['BMI_Category'] = pd.cut(df['BMI'],
                             bins=[0, 18.5, 25, 30, 35, 100],
                             labels=['Underweight','Normal','Overweight','Obese I','Obese II+'])

# Binary target
df['Cancer'] = (df['Appendix_Cancer_Prediction'] == 'Yes').astype(int)

# Encode categoricals for correlation
cat_cols = ['Gender','Smoking_Status','Alcohol_Consumption','Family_History_Cancer',
            'Genetic_Mutations','Chronic_Diseases','Physical_Activity_Level',
            'Diet_Type','Radiation_Exposure','Previous_Cancers','Tumor_Markers',
            'Symptom_Severity','Treatment_Type']

df_enc = df.copy()
for c in cat_cols:
    df_enc[c] = df_enc[c].astype('category').cat.codes

print('Feature engineering complete.')

---
## Visualization 1 — Cancer vs Non-Cancer Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Fig 1 — Target Class Distribution', fontsize=15, fontweight='bold')

counts = df['Appendix_Cancer_Prediction'].value_counts()
colors = [CANCER_COLORS[l] for l in counts.index]

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Count per Class')
axes[0].set_ylabel('Number of Patients')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{bar.get_height():,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, pctdistance=0.75,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig('viz1_target_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: viz1_target_distribution.png')

---
## Visualization 2 — Age & Gender Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Fig 2 — Age & Gender Distributions by Cancer Outcome', fontsize=15, fontweight='bold')

# 2a: Age histogram by cancer status
for label, grp in df.groupby('Appendix_Cancer_Prediction'):
    axes[0].hist(grp['Age'], bins=30, alpha=0.65,
                 color=CANCER_COLORS[label], label=f'Cancer={label}', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')
axes[0].legend()

# 2b: KDE
for label, grp in df.groupby('Appendix_Cancer_Prediction'):
    grp['Age'].plot.kde(ax=axes[1], color=CANCER_COLORS[label], linewidth=2.5, label=f'Cancer={label}')
axes[1].set_title('Age Density (KDE)')
axes[1].set_xlabel('Age (years)')
axes[1].legend()

# 2c: Gender breakdown stacked bar
gender_cancer = df.groupby(['Gender','Appendix_Cancer_Prediction']).size().unstack(fill_value=0)
gender_cancer_pct = gender_cancer.div(gender_cancer.sum(axis=1), axis=0) * 100
gender_cancer_pct.plot(kind='bar', ax=axes[2], color=[CANCER_COLORS['No'], CANCER_COLORS['Yes']],
                        edgecolor='white', rot=0)
axes[2].set_title('Cancer Rate by Gender')
axes[2].set_ylabel('Percentage (%)')
axes[2].set_xlabel('')
axes[2].legend(title='Cancer', labels=['No','Yes'])

plt.tight_layout()
plt.savefig('viz2_age_gender.png', bbox_inches='tight')
plt.show()
print('Saved: viz2_age_gender.png')

---
## Visualization 3 — Key Risk Factor Rates

In [ ]:
risk_factors = [
    'Smoking_Status', 'Alcohol_Consumption', 'Family_History_Cancer',
    'Genetic_Mutations', 'Radiation_Exposure', 'Previous_Cancers'
]

# Cancer rate for each positive flag in each risk factor
def cancer_rate_for_factor(col):
    return df.groupby(col)['Cancer'].mean().mul(100)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Fig 3 — Cancer Rate by Key Risk Factors', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flat, risk_factors):
    rates = cancer_rate_for_factor(col)
    bar_colors = ['#e05c5c' if v == rates.max() else '#5c9ae0' for v in rates.values]
    bars = ax.bar(rates.index.astype(str), rates.values, color=bar_colors, edgecolor='white', linewidth=1.2)
    ax.set_title(col.replace('_', ' '))
    ax.set_ylabel('Cancer Rate (%)')
    ax.set_ylim(0, rates.max() * 1.25)
    for bar, val in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.savefig('viz3_risk_factors.png', bbox_inches='tight')
plt.show()
print('Saved: viz3_risk_factors.png')

---
## Visualization 4 — BMI & Lifestyle Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Fig 4 — BMI & Lifestyle vs Cancer Outcome', fontsize=15, fontweight='bold')

# 4a: BMI box plot
df.boxplot(column='BMI', by='Appendix_Cancer_Prediction', ax=axes[0],
           patch_artist=True,
           boxprops=dict(facecolor='#dce8f5'),
           medianprops=dict(color='#e05c5c', linewidth=2.5))
axes[0].set_title('BMI by Cancer Status')
axes[0].set_xlabel('Cancer')
axes[0].set_ylabel('BMI')
plt.sca(axes[0])
plt.title('BMI by Cancer Status')

# 4b: BMI category cancer rate
bmi_rates = df.groupby('BMI_Category', observed=True)['Cancer'].mean().mul(100)
bmi_rates.plot(kind='bar', ax=axes[1], color='#e07a5c', edgecolor='white', rot=25)
axes[1].set_title('Cancer Rate by BMI Category')
axes[1].set_ylabel('Cancer Rate (%)')
axes[1].set_xlabel('')
for i, v in enumerate(bmi_rates.values):
    axes[1].text(i, v + 0.2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

# 4c: Physical activity vs cancer
activity_rates = df.groupby('Physical_Activity_Level')['Cancer'].mean().mul(100)
activity_rates.plot(kind='bar', ax=axes[2], color='#5ca87a', edgecolor='white', rot=0)
axes[2].set_title('Cancer Rate by Physical Activity')
axes[2].set_ylabel('Cancer Rate (%)')
axes[2].set_xlabel('')
for i, v in enumerate(activity_rates.values):
    axes[2].text(i, v + 0.2, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('viz4_bmi_lifestyle.png', bbox_inches='tight')
plt.show()
print('Saved: viz4_bmi_lifestyle.png')

---
## Visualization 5 — Correlation Heatmap

In [ ]:
num_cols = ['Age','BMI','Blood_Pressure','Cholesterol_Level',
            'White_Blood_Cell_Count','Red_Blood_Cell_Count','Platelet_Count',
            'Diagnosis_Delay_Days','Survival_Years_After_Diagnosis','Cancer']

corr_num = df_enc[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_num, dtype=bool))
sns.heatmap(corr_num, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Fig 5 — Correlation Heatmap (Numeric Features)', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('viz5_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved: viz5_correlation_heatmap.png')

---
## Visualization 6 — Top Countries by Cancer Rate

In [ ]:
country_stats = df.groupby('Country').agg(
    Total=('Cancer','count'),
    CancerRate=('Cancer','mean')
).reset_index()
country_stats['CancerRate'] *= 100

# Keep only countries with enough samples (>= 500)
country_stats = country_stats[country_stats['Total'] >= 500].sort_values('CancerRate')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Fig 6 — Geographic Analysis', fontsize=15, fontweight='bold')

# Top & bottom 12 countries
top12 = country_stats.tail(12)
bot12 = country_stats.head(12)

axes[0].barh(top12['Country'], top12['CancerRate'], color='#e05c5c', edgecolor='white')
axes[0].set_title('Top 12 Countries — Highest Cancer Rate')
axes[0].set_xlabel('Cancer Rate (%)')
for i, (_, row) in enumerate(top12.iterrows()):
    axes[0].text(row['CancerRate'] + 0.1, i, f"{row['CancerRate']:.1f}%", va='center', fontsize=8.5)

axes[1].barh(bot12['Country'], bot12['CancerRate'], color='#5c9ae0', edgecolor='white')
axes[1].set_title('Top 12 Countries — Lowest Cancer Rate')
axes[1].set_xlabel('Cancer Rate (%)')
for i, (_, row) in enumerate(bot12.iterrows()):
    axes[1].text(row['CancerRate'] + 0.05, i, f"{row['CancerRate']:.1f}%", va='center', fontsize=8.5)

plt.tight_layout()
plt.savefig('viz6_country_rates.png', bbox_inches='tight')
plt.show()
print('Saved: viz6_country_rates.png')

---
## Visualization 7 — Age Group x Cancer Rate (+ Gender split)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Fig 7 — Age Group Trends', fontsize=15, fontweight='bold')

# 7a: Cancer rate by age group
age_rates = df.groupby('Age_Group', observed=True)['Cancer'].mean().mul(100)
axes[0].plot(age_rates.index.astype(str), age_rates.values,
             marker='o', markersize=9, linewidth=2.5, color='#e05c5c')
axes[0].fill_between(range(len(age_rates)), age_rates.values, alpha=0.15, color='#e05c5c')
axes[0].set_title('Cancer Rate by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Cancer Rate (%)')
for i, v in enumerate(age_rates.values):
    axes[0].annotate(f'{v:.1f}%', (i, v), textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=9)

# 7b: Heatmap — age group x gender cancer rate
pivot = df.groupby(['Age_Group','Gender'], observed=True)['Cancer'].mean().mul(100).unstack()
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[1], cbar_kws={'label': 'Cancer Rate (%)'})
axes[1].set_title('Cancer Rate (%) — Age Group × Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Age Group')

plt.tight_layout()
plt.savefig('viz7_age_trends.png', bbox_inches='tight')
plt.show()
print('Saved: viz7_age_trends.png')

---
## Visualization 8 — Blood Markers & Tumor Markers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Fig 8 — Clinical Markers vs Cancer Outcome', fontsize=15, fontweight='bold')

blood_markers = ['White_Blood_Cell_Count','Red_Blood_Cell_Count','Platelet_Count','Cholesterol_Level']

for ax, col in zip(axes.flat, blood_markers):
    for label, grp in df.groupby('Appendix_Cancer_Prediction'):
        grp[col].plot.kde(ax=ax, color=CANCER_COLORS[label],
                          linewidth=2.2, label=f'Cancer={label}')
    ax.set_title(col.replace('_', ' '))
    ax.set_xlabel(col.replace('_', ' '))
    ax.legend()
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig('viz8_clinical_markers.png', bbox_inches='tight')
plt.show()
print('Saved: viz8_clinical_markers.png')

---
## 3. Statistical Tests — Key Features

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency

cancer_yes = df[df['Cancer'] == 1]
cancer_no  = df[df['Cancer'] == 0]

print('=== T-Tests (numeric features, Cancer vs No Cancer) ===')
for col in ['Age','BMI','White_Blood_Cell_Count','Red_Blood_Cell_Count',
             'Platelet_Count','Cholesterol_Level','Blood_Pressure','Diagnosis_Delay_Days']:
    t, p = ttest_ind(cancer_yes[col].dropna(), cancer_no[col].dropna())
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'  {col:<35} t={t:7.3f}  p={p:.4f}  {sig}')

print('\n=== Chi-Square Tests (categorical features vs Cancer) ===')
for col in ['Gender','Smoking_Status','Alcohol_Consumption','Family_History_Cancer',
            'Genetic_Mutations','Radiation_Exposure','Previous_Cancers','Tumor_Markers',
            'Symptom_Severity','Diet_Type','Physical_Activity_Level']:
    ct = pd.crosstab(df[col], df['Appendix_Cancer_Prediction'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'  {col:<35} chi2={chi2:9.2f}  p={p:.4f}  {sig}')

---
## 4. Feature Importance (Point-Biserial Correlation with Target)

In [ ]:
from scipy.stats import pointbiserialr

feature_cols = [c for c in df_enc.columns
                if c not in ['Patient_ID','Appendix_Cancer_Prediction','Cancer',
                              'Age_Group','BMI_Category']]

pb_corrs = {}
for col in feature_cols:
    try:
        r, p = pointbiserialr(df_enc[col].fillna(0), df_enc['Cancer'])
        pb_corrs[col] = (abs(r), r, p)
    except Exception:
        pass

pb_df = pd.DataFrame(pb_corrs, index=['abs_r','r','p']).T.sort_values('abs_r', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e05c5c' if r > 0 else '#5c9ae0' for r in pb_df['r'].values]
ax.barh(pb_df.index[::-1], pb_df['abs_r'][::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Feature Correlation with Cancer Outcome (|Point-Biserial r|)',
              fontsize=13, fontweight='bold')
ax.set_xlabel('|Correlation|')
plt.tight_layout()
plt.savefig('viz_feature_importance.png', bbox_inches='tight')
plt.show()
print('Saved: viz_feature_importance.png')

print('\nTop 10 features by correlation with Cancer:')
print(pb_df[['r','p']].head(10).to_string())

---
## 5. Key Insights Summary

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency

cancer_yes = df[df['Cancer'] == 1]
cancer_no  = df[df['Cancer'] == 0]
total = len(df)
cancer_pct = df['Cancer'].mean() * 100

# recompute helpers
country_stats_local = df.groupby('Country').agg(
    Total=('Cancer','count'), CancerRate=('Cancer','mean')).reset_index()
country_stats_local['CancerRate'] *= 100
country_stats_local = country_stats_local[country_stats_local['Total'] >= 500].sort_values('CancerRate')

ct_gender = pd.crosstab(df['Gender'], df['Appendix_Cancer_Prediction'])
chi2_gender, p_gender, _, _ = chi2_contingency(ct_gender)
t_age, p_age = ttest_ind(cancer_yes['Age'], cancer_no['Age'])

gender_rates = df.groupby('Gender')['Cancer'].mean().mul(100)
bmi_rates_s  = df.groupby('BMI_Category', observed=True)['Cancer'].mean().mul(100)
top5_c       = country_stats_local.tail(5)

print('=' * 68)
print('KEY INSIGHTS — Appendix Cancer Prediction EDA')
print('=' * 68)
print(f'\nDataset: {total:,} patients | Cancer-positive: {cancer_pct:.1f}% (1 in ~6.6)')
print('Class imbalance: ~85% No / ~15% Yes  →  use stratified splits in ML.')

print(f"""
INSIGHT 1 — CLASS IMBALANCE
  Cancer rate is {cancer_pct:.1f}% overall. Dataset is moderately imbalanced.
  ML models should apply class weighting or SMOTE to avoid majority-class bias.

INSIGHT 2 — AGE  (p={p_age:.3f} — NOT significant)
  Median age is identical in both groups (~53 yrs).
  Age alone does NOT distinguish cancer from non-cancer in this dataset.
  The uniform age spread is typical of a synthetically generated dataset.

INSIGHT 3 — GENDER  (chi²={chi2_gender:.2f}, p={p_gender:.4f} — ONLY significant feature)
  Female : {gender_rates.get('Female',0):.2f}%  |  Male : {gender_rates.get('Male',0):.2f}%  |  Other : {gender_rates.get('Other',0):.2f}%
  Gender is the only statistically significant predictor (p < 0.05).
  Effect size is small — not clinically actionable in isolation.

INSIGHT 4 — TRADITIONAL RISK FACTORS  (all p > 0.30 — NOT significant)
  Smoking, Alcohol, Family History, Genetic Mutations, Radiation,
  and Previous Cancers all show NO significant difference in cancer rate.
  Every group clusters tightly around the global baseline of ~15%.
  This is a hallmark of synthetic data where the label was generated
  independently of these variables.

INSIGHT 5 — BLOOD & CLINICAL MARKERS  (all p > 0.13 — NOT significant)
  WBC, RBC, Platelets, Cholesterol, and Blood Pressure distributions
  almost perfectly overlap between cancer/non-cancer groups.
  No individual blood marker separates the classes.

INSIGHT 6 — BMI
  Slight variation: peak cancer rate in '{bmi_rates_s.idxmax()}' category ({bmi_rates_s.max():.1f}%).
  Difference is not statistically significant across BMI groups.

INSIGHT 7 — GEOGRAPHY  (Top 5 highest-rate countries)""")
print(top5_c[['Country','CancerRate']].to_string(index=False))
print("""
  Country-level variation is narrow (~13–17%). No country is a strong
  outlier. Geographic patterns should be interpreted cautiously.

INSIGHT 8 — RECOMMENDATION FOR ML TEAM
  No single feature shows a strong univariate relationship with cancer
  outcome (max |r| < 0.006 across all features).
  The classifier must capture complex non-linear interactions.
  Recommended: XGBoost / Random Forest with full feature set +
  class_weight='balanced'. Expect modest AUC until feature interactions
  are explored via SHAP or permutation importance.""")
print('\n' + '=' * 68)


---
## 6. Deliverables Checklist

| # | Deliverable | File | Status |
|---|-------------|------|---------|
| 1 | EDA Notebook | `eda_jyotsana.ipynb` | Done |
| 2 | Target distribution | `viz1_target_distribution.png` | Done |
| 3 | Age & Gender distributions | `viz2_age_gender.png` | Done |
| 4 | Key risk factors | `viz3_risk_factors.png` | Done |
| 5 | BMI & Lifestyle | `viz4_bmi_lifestyle.png` | Done |
| 6 | Correlation heatmap | `viz5_correlation_heatmap.png` | Done |
| 7 | Country/geographic rates | `viz6_country_rates.png` | Done |
| 8 | Age group trends + heatmap | `viz7_age_trends.png` | Done |
| 9 | Clinical blood markers | `viz8_clinical_markers.png` | Done |
|10 | Feature importance chart | `viz_feature_importance.png` | Done |